Do a transfer analysis between behavior decoders estimated on early and late trial data.

In [ ]:
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from tqdm.auto import tqdm
import mne
import re

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.data import add_metadata_features

In [ ]:
subject = "EC248"

result_path = f"outputs/causal4/behavior_decoding_single_electrode/{subject}/results.pt"
groupby = ["word_end"]

A_early_final_summary_path = f"outputs/causal4/behavior_decoding_single_electrode_summarize/{subject}/A_early_final_summary.csv"
A_final_summary_path = f"outputs/causal4/behavior_decoding_single_electrode_summarize/{subject}/A_final_summary.csv"

# single-electrode stimulus decoding results
A_individual_results_path = f"outputs/causal4/find_As/{subject}_results.csv"

trf_results_path = "/userdata/jgauthier/projects/big-trf/outputs/encoder_summary/timit-no_repeats/vanilla_aud.csv"

epochs_path = f"outputs/epochs_preprocessed/{subject}_epo.fif"

phoneme_pair_order = ["bm", "dn", "pb"]

outdir = "."

In [ ]:
# subject may not have been updated in injected params; induce from path
subject = re.findall(r"/behavior_decoding_single_electrode/([^/]+)/", result_path)[0]

In [ ]:
epochs = mne.read_epochs(epochs_path, preload=True, verbose=False)
epochs.metadata = add_metadata_features(epochs.metadata)

In [ ]:
individual_A_results = pd.read_csv(A_individual_results_path).query("A")

In [ ]:
# Merge in TRF information for As
trf_results = pd.read_csv(trf_results_path)
individual_A_results = pd.merge(individual_A_results, trf_results.rename(columns={"output_dim": "electrode_idx"}).groupby(["subject", "electrode_idx"]).score.mean().rename("trf_r2"),
            on=["subject", "electrode_idx"], how="left")

In [ ]:
behav_decoder_result = torch.load(result_path)

A_decoding_results = behav_decoder_result["A_decoding_results"]
B_decoding_results = behav_decoder_result["B_decoding_results"]
C_decoding_results = behav_decoder_result["C_decoding_results"]

A_decoders = behav_decoder_result["A_decoders"]
B_decoders = behav_decoder_result["B_decoders"]
C_decoders = behav_decoder_result["C_decoders"]

In [ ]:
A_early_final_summary = pd.read_csv(A_early_final_summary_path)
A_final_summary = pd.read_csv(A_final_summary_path)

In [ ]:
comp_keep_keys = [
    "subject", "population", "phoneme_pair", "word_end", "smin", "smax",
    "baseline_roc_auc", "full_roc_auc", "diff"
]
A_comp_df = pd.merge(
    A_early_final_summary[comp_keep_keys],
    A_final_summary[comp_keep_keys],
    on=["subject", "population", "phoneme_pair", "word_end"],
    suffixes=("_early", "_late"))

In [ ]:
assert (A_comp_df.baseline_roc_auc_early == A_comp_df.baseline_roc_auc_late).all(), "sanity check"

In [ ]:
ax = sns.scatterplot(data=A_comp_df, x="diff_early", y="diff_late", hue="phoneme_pair")
ax.axvline(0, color="red", linestyle="--")
ax.axhline(0, color="red", linestyle="--")
# plot y=x, y=-x
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]
ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
ax.plot(lims, [-l for l in lims], 'k--', alpha=0.5, zorder=0)

ax.set_xlim((-0.035, 0.035))
ax.set_ylim((-0.005, 0.04))

ax.legend(title="Phoneme pair", title_fontsize=13, fontsize=11)

ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.set_xlabel("Increase in ROC-AUC\nover baseline\n(Early phonetic window)")
ax.set_ylabel("Increase in ROC-AUC\nover baseline\n(Phonetic feedback window)")

In [ ]:
from src.viz import spaghetti_plot as general_spaghetti

In [ ]:
g = general_spaghetti(
    A_comp_df.reset_index().rename(columns={"population": "electrode_idx"}).melt(
        id_vars=["subject", "electrode_idx", "phoneme_pair", "word_end"],
        value_vars=["diff_early", "diff_late"],),
    y="value", by="variable", by1="diff_early", by2="diff_late",
    col_order=phoneme_pair_order,
    sharey=False)
g.set_titles(col_template="{col_name}")
for ax in g.axes.flat:
    ax.axhline(0, color="red", linestyle="--")
    ax.set_xticks([])
    ax.set_xlabel(None)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))

handles = g.legend.legend_handles
labels = ["Phonetic", "Feedback"]
g.legend.remove()
g.axes.flat[-1].legend(title="Population\nkind", handles=handles, labels=labels,
                       fontsize=13, title_fontsize=13)

In [ ]:
### Do a direct transfer of behavior decoders from early -> late
transfer_results = []
num_folds = 5

def prepare_behavior_decoding_data(ep_i, data_i, epoch_idxs, electrode_idx, smin, smax,
                                   baseline_vars=["resampled"]):
    X = data_i[epoch_idxs][:, electrode_idx, smin:smax]  # shape (n_epochs, n_times)
    # add baseline information
    X_baseline = ep_i.metadata.loc[epoch_idxs][baseline_vars].values
    X = np.concatenate([X_baseline, X], axis=1)
    return X

for row in tqdm(A_comp_df.reset_index().itertuples(), total=len(A_comp_df)):
    ep_i = epochs
    md_i = ep_i.metadata
    data_i = ep_i.get_data()  # shape (n_epochs, n_channels, n_times)

    electrode_idx = int(row.population)
    key = (row.subject, electrode_idx, row.phoneme_pair)
    early_key = (row.subject, str(electrode_idx), row.phoneme_pair, (row.word_end,), row.smin_early, row.smax_early)
    late_key = (row.subject, str(electrode_idx), row.phoneme_pair, (row.word_end,), row.smin_late, row.smax_late)

    early_pipes, early_dec_outcomes = {}, {}

    # First reproduce the original results -- validate that we can get the exact
    # same behavioral predictions using the saved decoders
    for early_fold in range(num_folds):
        try:
            early_pipe = A_decoders[key][early_key + (early_fold,)]["estimator"]
        except KeyError:
            # no decoder for this fold; it didn't converge
            continue

        early_previous_outcomes = A_decoders[key][early_key + (early_fold,)]["test_predictions"]
        early_idxs = early_previous_outcomes.epoch_idx

        X = prepare_behavior_decoding_data(ep_i, data_i, early_idxs, electrode_idx,
                                           row.smin_early, row.smax_early)
        preds = early_pipe.predict_proba(X)[:, 1]
        np.testing.assert_allclose(preds, early_previous_outcomes.full_decoder_proba.values)

        early_pipes[early_fold] = early_pipe
        early_dec_outcomes[early_fold] = early_previous_outcomes

    # Now apply the saved early decoders to the late window data
    for early_fold in early_pipes.keys():
        early_pipe = early_pipes[early_fold]
        early_previous_outcomes = early_dec_outcomes[early_fold]

        # Get the companion late decoder
        late_pipe = A_decoders[key][late_key + (early_fold,)]["estimator"]
        late_previous_outcomes = A_decoders[key][late_key + (early_fold,)]["test_predictions"]
        late_idxs = late_previous_outcomes.epoch_idx
        early_idxs = early_previous_outcomes.epoch_idx
        assert np.array_equal(early_idxs, late_idxs), "epoch indices should match between early and late folds"

        # Apply late decoder to early time window
        X_early = prepare_behavior_decoding_data(ep_i, data_i, early_idxs, electrode_idx,
                                                 row.smin_early, row.smax_early)
        late_on_early_preds = late_pipe.predict_proba(X_early)[:, 1]

        # 20251124: Not necessary
        # # Try manually demeaning early window too. This is because we know that early activations
        # # are of a way different magnitude.
        # X_early_demeaned = X_early.copy()
        # X_early_demeaned -= X_early_demeaned.mean(axis=0, keepdims=True)
        # late_on_early_demeaned_preds = late_pipe.predict_proba(X_early_demeaned)[:, 1]

        # Apply early decoder to late time window
        X_late = prepare_behavior_decoding_data(ep_i, data_i, late_idxs, electrode_idx,
                                                row.smin_late, row.smax_late)
        early_on_late_preds = early_pipe.predict_proba(X_late)[:, 1]

        # # DEV: validate that late-on-late predictions match the earlier outcomes
        # late_on_late_preds = late_pipe.predict_proba(X_late)[:, 1]
        # np.testing.assert_allclose(late_on_late_preds, late_previous_outcomes.full_decoder_proba.values)

        df = pd.DataFrame({
            "subject": row.subject,
            "electrode_idx": electrode_idx,
            "phoneme_pair": row.phoneme_pair,
            "smin_early": row.smin_early,
            "smax_early": row.smax_early,
            "smin_late": row.smin_late,
            "smax_late": row.smax_late,
            "fold": early_fold,
            
            # early decoder applied to late time window
            "early_on_late_decoder_proba": early_on_late_preds,

            # late decoder applied to early time window
            "late_on_early_decoder_proba": late_on_early_preds,

            # early decoder on early time window (for comparison)
            "early_on_early_decoder_proba": early_previous_outcomes.full_decoder_proba.values,

            # late decoder on late time window (for comparison)
            "late_on_late_decoder_proba": late_previous_outcomes.full_decoder_proba.values,
            
            "early_baseline_decoder_proba": early_previous_outcomes.baseline_decoder_proba.values,
            "late_baseline_decoder_proba": late_previous_outcomes.baseline_decoder_proba.values,
            "decoder_target": late_previous_outcomes.decoder_target.values,
            "epoch_idx": early_idxs,
            "word_end": row.word_end,
        })

        transfer_results.append(df)

transfer_results_df = pd.concat(transfer_results, ignore_index=True)

In [ ]:
transfer_group_results = []
for (subject, electrode_idx, phoneme_pair, word_end, smin_early, smax_early, smin_late, smax_late), group in tqdm(transfer_results_df.groupby(["subject", "electrode_idx", "phoneme_pair", "word_end", "smin_early", "smax_early", "smin_late", "smax_late"])):
    roc_aucs = []
    for fold, group_i in group.groupby("fold"):
        from sklearn.metrics import roc_auc_score
        transfer_group_results.append({
            "subject": subject,
            "electrode_idx": electrode_idx,
            "phoneme_pair": phoneme_pair,
            "word_end": word_end,
            "smin_early": smin_early,
            "smax_early": smax_early,
            "smin_late": smin_late,
            "smax_late": smax_late,
            "fold": fold,
            "early_on_late_roc_auc": roc_auc_score(group_i.decoder_target, group_i.early_on_late_decoder_proba),
            "late_on_early_roc_auc": roc_auc_score(group_i.decoder_target, group_i.late_on_early_decoder_proba),
            # "late_on_early_demeaned_roc_auc": roc_auc_score(group_i.decoder_target, group_i.late_on_early_demeaned_decoder_proba),
            "early_on_early_roc_auc": roc_auc_score(group_i.decoder_target, group_i.early_on_early_decoder_proba),
            "late_on_late_roc_auc": roc_auc_score(group_i.decoder_target, group_i.late_on_late_decoder_proba),

            "early_baseline_roc_auc": roc_auc_score(group_i.decoder_target, group_i.early_baseline_decoder_proba),
            "late_baseline_roc_auc": roc_auc_score(group_i.decoder_target, group_i.late_baseline_decoder_proba),
        })

In [ ]:
transfer_group_results_df = pd.DataFrame(transfer_group_results)
transfer_group_results_df["early_on_late_diff"] = transfer_group_results_df.early_on_late_roc_auc - transfer_group_results_df.late_baseline_roc_auc
transfer_group_results_df["late_on_early_diff"] = transfer_group_results_df.late_on_early_roc_auc - transfer_group_results_df.early_baseline_roc_auc
# transfer_group_results_df["late_on_early_demeaned_diff"] = transfer_group_results_df.late_on_early_demeaned_roc_auc - transfer_group_results_df.early_baseline_roc_auc
transfer_group_results_df["early_on_early_diff"] = transfer_group_results_df.early_on_early_roc_auc - transfer_group_results_df.early_baseline_roc_auc
transfer_group_results_df["late_on_late_diff"] = transfer_group_results_df.late_on_late_roc_auc - transfer_group_results_df.late_baseline_roc_auc

In [ ]:
# take means over folds
transfer_group_results_df = transfer_group_results_df.groupby(["subject", "electrode_idx", "phoneme_pair", "word_end"]).mean().reset_index()
transfer_group_results_df = transfer_group_results_df.drop(columns=["fold"])

In [ ]:
# Merge in original phonetic decoding information
transfer_group_results_df = pd.merge(
    transfer_group_results_df,
    individual_A_results[["subject", "electrode_idx", "phoneme_pair", "roc_auc", "trf_r2"]].rename(columns={"roc_auc": "phonetic_roc_auc"}),
    on=["subject", "electrode_idx", "phoneme_pair"], how="left", validate="m:1")

In [ ]:
g = sns.catplot(data=transfer_group_results_df.reset_index()
                .melt(id_vars=["subject", "electrode_idx", "phoneme_pair", "word_end"], 
                      value_vars=["early_on_late_diff", "late_on_early_diff", "early_on_early_diff", "late_on_late_diff"]),
            x="variable", y="value", kind="box", col="phoneme_pair", showfliers=False, height=8, aspect=0.5)
for ax in g.axes.flat:
    ax.axhline(0, color="red", linestyle="--")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

In [ ]:
g = sns.catplot(
    data=transfer_group_results_df.reset_index().rename(columns={"late_on_late_diff": "Feedback", "early_on_late_diff": "Phonetic"})
                .melt(id_vars=["subject", "electrode_idx", "phoneme_pair", "word_end"], value_vars=["Feedback", "Phonetic"]), 
            x="variable", y="value", kind="box", order=["Phonetic", "Feedback"],
            height=4, aspect=0.8, col="phoneme_pair", showfliers=False, col_order=phoneme_pair_order)
g.set_axis_labels("Evaluation time window", "Behavior decoding\nROC-AUC")
for i, ax in enumerate(g.axes.flat):
    ax.axhline(0, color="red", linestyle="--")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    if i != 1:
        ax.set_xlabel(None)

g.set_titles(col_template="{col_name}")

In [ ]:
ttest_rel(transfer_group_results_df.late_on_late_diff, transfer_group_results_df.early_on_late_diff)

In [ ]:
transfer_group_results_df.to_csv(Path(outdir) / f"transfer_results.csv", index=False)